# Resultados do BVTSLD

Este notebook apenas inspeciona artefatos produzidos pelos scripts. A geração de dados, as seleções e os treinos permanecem no pipeline retomável `scripts/reproduce.py`. Execute as células em ordem.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUTPUT = ROOT / 'outputs' / 'bvtsld'
assert (OUTPUT / 'selections_summary.csv').exists(), 'Execute a partir da raiz do projeto ou de notebooks/'

## Oráculo e seleções congeladas

In [ ]:
oracle = json.loads((OUTPUT / 'oracle_results.json').read_text())
selections = pd.read_csv(OUTPUT / 'selections_summary.csv')
print('Oráculo mAP@0.5:0.95:', oracle['validation']['map50_95'])
print('Seleções:', len(selections))
selections.groupby(['technique', 'fraction'])[['coverage_mean', 'coverage_max']].mean().round(4)

## Grade de treino

A célula abaixo funciona antes ou depois da grade completa. Antes dela, apenas informa que os resultados ainda não existem.

In [ ]:
results_path = OUTPUT / 'triage_results.csv'
if not results_path.exists():
    print('Grade ainda não executada. Use: python scripts/reproduce.py --stage train')
    results = None
else:
    results = pd.read_csv(results_path)
    print(f'Runs concluídos: {len(results)}/328')
    display(results.groupby(['technique', 'fraction'])[['map50', 'map50_95']].agg(['mean', 'std']).round(4))

In [ ]:
if results is not None:
    plot_data = results.groupby(['technique', 'fraction'])['map50_95'].mean().unstack(0)
    ax = plot_data.plot(marker='o', figsize=(9, 5))
    ax.axhline(oracle['validation']['map50_95'], color='black', linestyle='--', label='oráculo')
    ax.set(xlabel='Fração rotulada', ylabel='mAP@0.5:0.95', title='Desempenho por orçamento de rótulos')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()

## Inferência estatística

Após os 328 runs, gere os testes pareados com `python scripts/reproduce.py --stage analyze`. O arquivo resultante é `outputs/bvtsld/triage_analysis.csv`.

In [ ]:
analysis_path = OUTPUT / 'triage_analysis.csv'
pd.read_csv(analysis_path) if analysis_path.exists() else 'Análise ainda não gerada.'